# 1. DATASET PREPARATION

In [ ]:
!unzip /content/images.zip # unzio the data

unzip:  cannot find or open /content/images.zip, /content/images.zip.zip or /content/images.zip.ZIP.


In [ ]:
!mkdir /content/dataset # create a folder

mkdir: cannot create directory ‘/content/dataset’: File exists


In [ ]:
!mkdir /content/datasetimages # create another folder

In [ ]:
!mv /content/images.jpg /content/datasetimages # copy the image to folder created

mv: cannot stat '/content/images.jpg': No such file or directory


In [ ]:
!mv /content/datasetimages /content/dataset # move the folder to dataset folder

mv: cannot overwrite '/content/dataset/datasetimages': Directory not empty


# 2. IMPORTS AND SETUP

In [ ]:
import torch
import torch.nn as nn #deap learning lib cn, l, cg
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

 # 3. DATA TRANSFORMATIONS

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),

    transforms.ToTensor()
])

# 4. CUSTOM DATASET (SINGLE CLASS)

In [ ]:
class SingleClassDataset(torch.utils.data.Dataset):
    def __init__(self, folder, transform):
        self.paths = [os.path.join(folder, f) for f in os.listdir(folder)]
        self.transform = transform #SAVE ALL THE IMAGE FILE PATHS
        #SAVED TRANSFORM PIPELINE FOR THE LATER USE

    def __len__(self):
        return len(self.paths) #RETURN NUMBER OF THE IMAGE IN DATSET

    def __getitem__(self, idx):
        from PIL import Image
        img = Image.open(self.paths[idx]).convert("RGB") #OPEN THE IMAGE AND CONVERT THSAT IN TO rgb
        return self.transform(img), 0  # single class label

In [ ]:
dataset = SingleClassDataset("/content/dataset/datasetimages", transform)
loader = DataLoader(dataset, batch_size=1, shuffle=True) #one image at a time

# 5. MODEL DEFINITION ( CNN)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(), #non linearity
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(), #again activation
            nn.Flatten(),
            nn.Linear(32 * 128 * 128, 10)
        )

    def forward(self, x):
        return self.model(x) #define the foward pass though the CNN
# create model and move it to the selected device
model = SimpleCNN().to(device)

# 6. LOSS FUNCTION AND OPTIMIZER

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 7. LOAD SINGLE IMAGE SAMPLE

In [ ]:

#data_iter = iter(loader)
#images, labels = next(data_iter)

images, labels = dataset[0] #to load the image
images = images.unsqueeze(0).to(device) #add batch dimention moves the image to cpu or gpu
labels = torch.tensor([labels]).to(device) #converting the labels in to tensor and move in to device

images = images.to(device)
labels = labels.to(device)

UnidentifiedImageError: cannot identify image file '/content/dataset/datasetimages/images.zip'

# 8. FORWARD + BACKWARD PASS

In [ ]:
# Forward
outputs = model(images) #send imag though the CNN
loss = criterion(outputs, labels) # loss function inorder to compute gradients

# Backward
optimizer.zero_grad() #initial optimiser 0
loss.backward()

# 9. SAVE GRADIENTS

In [ ]:
# SAVE GRADIENTS
gradients = []
for param in model.parameters():
    gradients.append(param.grad.clone().detach()) #copy whole the saved gradients in to
    #in to the list

print("Gradients saved.")

# 10. GRADIENT INVERSION (RECONSTRUCTION)

In [ ]:
dummy_data = torch.randn_like(images, requires_grad=True, device=device) #random noise image
dummy_label = torch.tensor([0], device=device) #dummy image label

optimizer_dummy = optim.Adam([dummy_data], lr=0.1) #optimizing rhe dummy image

for i in range(250):  # iterations
    optimizer_dummy.zero_grad() #clear old dummy gradients

    pred = model(dummy_data) #send dummy image through CNN
    loss_dummy = criterion(pred, dummy_label) #compute loss fo the dummy image

    dummy_grads = torch.autograd.grad(loss_dummy, model.parameters(), create_graph=True) #generate gradeitns from dummy image

    grad_diff = 0 #starts the total gradient matching
    for g1, g2 in zip(dummy_grads, gradients):
        grad_diff += ((g1 - g2) ** 2).sum()

    grad_diff.backward()
    optimizer_dummy.step() #update dummy image so it become closer to orginal

    if i % 50 == 0:
        print(f"Iteration {i}, Loss: {grad_diff.item()}") #prrint progress every 50 iteration

# 11. VISUALIZATION FUNCTION

In [ ]:
def imshow(tensor, title): #display image tensor
    img = tensor.detach().cpu().squeeze().permute(1, 2, 0) #prepare tensor for plotting
    plt.imshow(img) #show image
    plt.title(title) #title
    plt.axis("off") #hide axes

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
imshow(images, "Original")

plt.subplot(1,2,2)
imshow(dummy_data, "Reconstructed")

plt.show()

# 13. METRICS (MSE, PSNR, SSIM)

In [ ]:
import torch.nn.functional as F
import math #MSE PIXEL BETWEEN TWWO IMAGE

def compute_metrics(original, reconstructed): #calculate the metrics

    orig = original.detach() #detached version of the orginal image
    recon = reconstructed.detach() #detach reconstrcuetd image tensor

    # MSE
    mse = F.mse_loss(recon, orig).item() #caalculate MSE between the orginal anf the reconstcuted

    # PSNR NOISE
    if mse == 0: #check the perfect reconstrcution
        psnr = 100 #assign a very higher value for perfect reconstcution
    else:
        psnr = 20 * math.log10(1.0 / math.sqrt(mse))  # assuming image range [0,1]

    return mse, psnr

In [ ]:
from skimage.metrics import structural_similarity as ssim #for structurral image comparison
import numpy as np #array based image representation

def compute_ssim(original, reconstructed): #to calculate the ssim in between og and rec
    orig = original.detach().cpu().squeeze().permute(1,2,0).numpy() #convert orginal image tensor in to numoy image format
    recon = reconstructed.detach().cpu().squeeze().permute(1,2,0).numpy() #saame for the rec

    score = ssim(orig, recon, channel_axis=2, data_range=1.0)
    return score

# 14. DISPLAY RESULTS WITH METRICS

In [ ]:
mse, psnr = compute_metrics(images, dummy_data) #psnr and the mse for the og vs rec

try:
    ssim_score = compute_ssim(images, dummy_data) #calculate ssim between the rec vs og
except:
    ssim_score = None #if it fails

def imshow(tensor, title):
    img = tensor.detach().cpu().squeeze().permute(1, 2, 0)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(10,4)) #w,h

plt.subplot(1,2,1)#r1,c2,1st
imshow(images, "Original")

title_recon = f"Reconstructed\nMSE: {mse:.6f}\nPSNR: {psnr:.2f} dB"
if ssim_score is not None:
    title_recon += f"\nSSIM: {ssim_score:.4f}"

plt.subplot(1,2,2)
imshow(dummy_data, title_recon)

plt.show()

# 15. BLURRED IMAGE EXPERIMENT

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) #sgrda

#data_iter = iter(loader)
#images, labels = next(data_iter)

images = images.to(device)
labels = labels.to(device)

# BLUR IMAGE BEFORE GRADIENT COMPUTATION
import torchvision.transforms.functional as TF
blurred_images = TF.gaussian_blur(images, kernel_size=11, sigma=5)

# Forward
outputs = model(blurred_images)
loss = criterion(outputs, labels)

# Backward
optimizer.zero_grad()
loss.backward()



# 16. SAVE GRADIENTS (BLURRED INPUT)

In [ ]:
# SAVE GRADIENTS
gradients = []
for param in model.parameters():
    gradients.append(param.grad.clone().detach())

print("Gradients saved from blurred input.")

# 17. RECONSTRUCTION FROM BLURRED GRADIENTS

In [ ]:
dummy_data = torch.randn_like(images, requires_grad=True, device=device) #create a random starting for reconstruction
dummy_label = torch.tensor([0], device=device) #labeling loss for the dummy

optimizer_dummy = optim.Adam([dummy_data], lr=0.1) #optimisation

for i in range(250):  # iterations
    optimizer_dummy.zero_grad() #clear

    pred = model(dummy_data) #send dummy
    loss_dummy = criterion(pred, dummy_label) #compute loss

    dummy_grads = torch.autograd.grad(loss_dummy, model.parameters(), create_graph=True) #generae grad

    grad_diff = 0 #matching loss at zero
    for g1, g2 in zip(dummy_grads, gradients): #compare
        grad_diff += ((g1 - g2) ** 2).sum() # add squared difference

    grad_diff.backward() #matching loss
    optimizer_dummy.step() #update the the dummy

    if i % 50 == 0: #print every 50 intervwals
        print(f"Iteration {i}, Loss: {grad_diff.item()}")

# 18. FINAL VISUALIZATION (BLURRED CASE)

In [ ]:
def imshow(tensor, title):
    img = tensor.detach().cpu().squeeze().permute(1, 2, 0) #to a image
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(8,4)) #side

plt.subplot(1,2,1)  #1 orginal
imshow(images, "Original")

plt.subplot(1,2,2) #2 re
imshow(dummy_data, "Reconstructed")

plt.show()

In [ ]:
import torch.nn.functional as F
import math

def compute_metrics(original, reconstructed):
    # Ensure same shape
    orig = original.detach()
    recon = reconstructed.detach()

    # MSE
    mse = F.mse_loss(recon, orig).item()

    # PSNR
    if mse == 0:
        psnr = 100
    else:
        psnr = 20 * math.log10(1.0 / math.sqrt(mse))  # assuming image range [0,1]

    return mse, psnr

In [ ]:
from skimage.metrics import structural_similarity as ssim
import numpy as np

def compute_ssim(original, reconstructed):
    orig = original.detach().cpu().squeeze().permute(1,2,0).numpy()
    recon = reconstructed.detach().cpu().squeeze().permute(1,2,0).numpy()

    score = ssim(orig, recon, channel_axis=2, data_range=1.0)
    return score

In [ ]:
mse, psnr = compute_metrics(images, dummy_data)

try:
    ssim_score = compute_ssim(images, dummy_data)
except:
    ssim_score = None

def imshow(tensor, title):
    img = tensor.detach().cpu().squeeze().permute(1, 2, 0)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
imshow(images, "Original")

title_recon = f"Reconstructed\nMSE: {mse:.6f}\nPSNR: {psnr:.2f} dB"
if ssim_score is not None:
    title_recon += f"\nSSIM: {ssim_score:.4f}"

plt.subplot(1,2,2)
imshow(dummy_data, title_recon)

plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')